In [53]:
import pandas as pd
import numpy as np

In [54]:
historical_fires = pd.read_csv("../data/historical_fires.csv")
cimis = pd.read_csv("../data/cimis_merged.csv")
drought_index = pd.read_csv("../data/drought_index_merged.csv")
population = pd.read_csv("../data/population_merged.csv")

We are going to clean each one of them individually in order of how it was listed earlier. 

# Historical Fires

In [55]:
historical_fires['County'] = historical_fires['County'].str.lower().str.strip()
historical_fires["Date"] = pd.to_datetime(historical_fires["Date"], errors="coerce")
historical_fires = historical_fires.dropna(subset=["Date"])

historical_fires = historical_fires.dropna(subset = ["incident_acres_burned"])
historical_fires = historical_fires[historical_fires["incident_acres_burned"] > 0]

In [56]:
# We can figure out California's Latitude and Longitude and constrict them to the range of California. This will help us filter out any fires that are not in California.   
valid_coords = (
    historical_fires["incident_latitude"].between(32, 42) &
    historical_fires["incident_longitude"].between(-124, -114)
)

historical_fires = historical_fires[valid_coords].copy()
historical_fires = historical_fires.drop(columns=["incident_type"])

historical_fires["Year"]  = historical_fires["Date"].dt.year
historical_fires["Month"] = historical_fires["Date"].dt.month
historical_fires["Day"]   = historical_fires["Date"].dt.day


# CIMIS 

In [57]:
cimis["County"] = cimis["County"].str.lower().str.strip()
cimis["Date"] = pd.to_datetime(cimis["Date"], format="%m/%d/%Y", errors="coerce")
cimis = cimis.dropna(subset=["Date"])
weather_columns = [
    "ETo (in)", "Precip (in)", "Sol Rad (Ly/day)", "Avg Vap Pres (mBars)",
    "Max Air Temp (F)", "Min Air Temp (F)", "Max Rel Hum (%)",
    "Avg Wind Speed (mph)", "Wind Run (miles)", "Avg Soil Temp (F)"
]

cimis = cimis.sort_values(['County', 'Date']).reset_index(drop=True)

In [58]:
interpolated_parts = []
for county_name, group in cimis.groupby("County"):
    group = group.copy().set_index("Date")
    group[weather_columns] = group[weather_columns].interpolate(
        method="time", limit=7, limit_direction="forward"
    )
    group = group.reset_index()
    group["County"] = county_name
    interpolated_parts.append(group)

cimis = pd.concat(interpolated_parts, ignore_index=True)
missing_cimis = set(historical_fires["County"].unique()) - set(cimis["County"].unique())

In [59]:
'''
To deal with missing counties, we can assign counties that have null values to the nearest neighboring county that has data. 
This is a common technique in spatial data analysis to handle missing values. 
'''
NEIGHBOR_MAP = {
    "lake":      "mendocino",
    "mariposa":  "madera",
    "calaveras": "amador",
    "tuolumne":  "amador",
    "trinity":   "tehama",
    "nevada":    "placer",
    "glenn":     "colusa",
    "mono":      "inyo",
    "sierra":    "plumas",
}

neighbor_dfs = []
for missing_county, neighbor in NEIGHBOR_MAP.items():
    if neighbor in cimis["County"].values:
        sub = cimis[cimis["County"] == neighbor].copy()
        sub["County"] = missing_county
        neighbor_dfs.append(sub)

if neighbor_dfs:
    cimis = pd.concat([cimis] + neighbor_dfs, ignore_index=True)


In [60]:
WEATHER_COLS = [
    "ETo (in)", "Precip (in)", "Sol Rad (Ly/day)", "Avg Vap Pres (mBars)",
    "Max Air Temp (F)", "Min Air Temp (F)", "Max Rel Hum (%)",
    "Avg Wind Speed (mph)", "Wind Run (miles)", "Avg Soil Temp (F)"
]
ROLLING_SPEC = {
    "Max Air Temp (F)":      ["mean", "max"],
    "Min Air Temp (F)":      ["mean", "min"],
    "Precip (in)":           ["sum",  "max"],
    "Avg Wind Speed (mph)":  ["mean", "max"],
    "Max Rel Hum (%)":       ["mean", "min"],
    "Sol Rad (Ly/day)":      ["mean"],
    "Avg Vap Pres (mBars)":  ["mean"],
    "ETo (in)":              ["sum"],
    "Avg Soil Temp (F)":     ["mean"],
    "Wind Run (miles)":      ["sum"],
}

# We use a CLOSED='left' rolling window so day T uses days [T-7, T-1]
# i.e. the 7 days BEFORE the fire date — no same-day leakage
WINDOW = 7

rolling_parts = []

for county_name, group in cimis.groupby("County"):
    g = group.set_index("Date").sort_index()

    new_cols = {}
    for col, aggs in ROLLING_SPEC.items():
        if col not in g.columns:
            continue
        series = g[col]
        # shift(1) then rolling(7) = window of [t-7, t-1] — excludes current day
        shifted = series.shift(1)
        roll    = shifted.rolling(window=WINDOW, min_periods=3)

        for agg in aggs:
            colname = col.replace(" (F)", "").replace(" (in)", "").replace(" (%)", "") \
                         .replace(" (mph)", "").replace(" (Ly/day)", "").replace(" (mBars)", "") \
                         .replace(" (miles)", "").replace(" ", "_")
            colname = f"{colname}_{agg}_7d"
            if agg == "mean": new_cols[colname] = roll.mean()
            elif agg == "max":  new_cols[colname] = roll.max()
            elif agg == "min":  new_cols[colname] = roll.min()
            elif agg == "sum":  new_cols[colname] = roll.sum()

    # Also add current-day "snapshot" features (the weather ON fire day)
    for col in WEATHER_COLS:
        if col in g.columns:
            new_cols[col] = g[col]

    result = pd.DataFrame(new_cols)
    result["County"] = county_name
    result = result.reset_index()  # brings Date back as column
    rolling_parts.append(result)

cimis_rolled = pd.concat(rolling_parts, ignore_index=True)

rolling_cols = [c for c in cimis_rolled.columns
                if c.endswith("_7d")]

# Drought Index

In [61]:
drought_index["County"] = drought_index["County"].str.lower().str.strip()
drought_index = drought_index.rename(columns={"None": "No_Drought", "ValidStart_Date": "Week_Start"})
drought_index["Week_Start"] = pd.to_datetime(drought_index["Week_Start"], errors="coerce")
drought_index = drought_index.dropna(subset=["Week_Start"])

To properly understand the drought index, we need to figure out what the D0 - D4 values represent. 
The values themselves are cumulative so D0 means D0 or worse while 

In [62]:
drought_index['Drought_Score']= (
    drought_index["D0"] + drought_index["D1"] + drought_index['D2'] + drought_index["D3"] + drought_index["D4"]
)
drought_index["Drought_D3_D4"] = drought_index["D3"] + drought_index["D4"]
drought_index["Any_Drought"]   = 100 - drought_index["No_Drought"]

drought_index = drought_index.sort_values(["County", "Week_Start"]).reset_index(drop=True)
drought_cols = ["No_Drought", "D0", "D1", "D2", "D3", "D4",
                "Drought_Score", "Drought_D3_D4", "Any_Drought"]

In [63]:
all_counties = drought_index["County"].unique()
fire_date_min = historical_fires["Date"].min()
fire_date_max = historical_fires["Date"].max()
daily_date_range = pd.date_range(fire_date_min, fire_date_max, freq="D")

expanded = []
for county in all_counties:
    sub = drought_index[drought_index["County"] == county][["Week_Start"] + drought_cols].copy()
    sub = sub.set_index("Week_Start").reindex(daily_date_range)
    sub = sub.ffill().bfill()        
    sub.index.name = "Date"
    sub["County"] = county
    expanded.append(sub.reset_index())

drought_daily = pd.concat(expanded, ignore_index=True)

# Population

In [64]:
population['County'] = population['County'].str.lower().str.strip()

# Merging
Now that every dataset has been processed, we can begin merging them into a single, holistic way of accessing information.

In [65]:
df = historical_fires.merge(
    cimis_rolled[["County", "Date"] + weather_columns + rolling_cols],
    on=["County", "Date"],
    how="left"
)

matched = df[weather_columns[0]].notna().sum()

df = df.merge(
    drought_daily[["County", 'Date'] + drought_cols],  
    on=["County", "Date"],
    how="left"
)
df = df.merge(
    population[["County", "Year", "Population", "Density", "Land Area (mi)"]],
    on=["County", "Year"],
    how="left"
)

pre_drop = len(df)
df = df.dropna(subset=["Max Air Temp (F)"])



# Feature Engineering

In [66]:
df["Temp_Range"]   = df["Max Air Temp (F)"] - df["Min Air Temp (F)"]
df["Avg_Temp"]     = (df["Max Air Temp (F)"] + df["Min Air Temp (F)"]) / 2

In [67]:
# Core CA fire season: May–October
df["Is_Fire_Season"] = df["Month"].between(5, 10).astype(int)


In [68]:
season_map = {12: "Winter", 1: "Winter", 2: "Winter",
              3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer",
              9: "Autumn", 10: "Autumn", 11: "Autumn"}
df["Season"] = df["Month"].map(season_map)

In [69]:
# log transform acres burned to handle skewness and create a more normalized distribution
df["log_acres_burned"] = np.log1p(df["incident_acres_burned"])

# Calculate fire impact per capita
df["Fire_Impact_Per_Capita"] = df["incident_acres_burned"] / df["Population"].replace(0, np.nan)

In [70]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
for col in num_cols:
    if df[col].isna().sum() > 0:
        county_medians = df.groupby("County")[col].transform("median")
        df[col] = df[col].fillna(county_medians)
        df[col] = df[col].fillna(df[col].median())

df = df.drop(columns=["incident_longitude", "incident_latitude"], errors="ignore")
df = df.drop(columns = [c for c in ['None'] if c in df.columns])

In [71]:
id_cols      = ["County", "Date", "Year", "Month", "Day"]
target_cols  = ["incident_acres_burned", "log_acres_burned"]
weather      = weather_columns
drought_out  = ["No_Drought", "D0", "D1", "D2", "D3", "D4",
                "Drought_Score", "Drought_D3_D4", "Any_Drought"]
engineered = ["Temp_Range", "Avg_Temp", "Is_Fire_Season", "Season",
              "Fire_Impact_Per_Capita"] + rolling_cols
pop_cols     = ["Population", "Density", "Land Area (mi)"]

final_cols = (id_cols + target_cols + weather + 
              [c for c in drought_out if c in df.columns] +
              engineered + pop_cols)
final_cols = [c for c in final_cols if c in df.columns]

extra = [c for c in df.columns if c not in final_cols]
df = df[final_cols + extra]

df = df.sort_values(["Date", "County"]).reset_index(drop=True)


In [72]:
df.to_csv("../data/final_dataset.csv", index=False)
